# Stage 3 (step-by-step): task distillation outline

Stage 3 adds **teacher vs student** logits, **KL**, **prefix consistency**, and **revision** losses (see `adapter_task_trainer.py`).
This notebook only **imports the adapter components** you listed; the heavy teacher/student loop stays in the trainer script pattern.

Use **`StreamingAdapter` + `EarlyCommitGate`** the same way as stage 2; losses are computed on stacked LLM forwards.


In [1]:
import os
import sys

os.environ.setdefault("TRANSFORMERS_VERBOSITY", "info")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

_nb = os.getcwd()
if os.path.basename(_nb) == "notebooks":
    PROJECT_ROOT = os.path.abspath(os.path.join(_nb, ".."))
else:
    PROJECT_ROOT = os.environ.get(
        "AUDIO_STREAM_ADAPTER_ROOT",
        os.path.abspath(os.path.join(_nb, "..")),
    )

SRC_ROOT = os.path.join(PROJECT_ROOT, "src")
TRAINING_DIR = os.path.join(PROJECT_ROOT, "training")

if not os.path.isdir(os.path.join(SRC_ROOT, "adapter")):
    raise FileNotFoundError(f"Expected package at {SRC_ROOT}/adapter — open notebook from notebooks/ or set AUDIO_STREAM_ADAPTER_ROOT")

for _p in (SRC_ROOT, PROJECT_ROOT):
    if _p not in sys.path:
        sys.path.insert(0, _p)

os.chdir(TRAINING_DIR)
os.makedirs(os.path.join(PROJECT_ROOT, "checkpoints"), exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("TRAINING_DIR:", TRAINING_DIR)
print("cwd:", os.getcwd())


PROJECT_ROOT: /home/ml/workspaces/kristina/audio-stream/audio-streaming-adapter
cwd (checkpoints here): /home/ml/workspaces/kristina/audio-stream/audio-streaming-adapter/src/adapter/training


In [ ]:
# Only these adapter modules (no stage*_trainer imports)
import torch
import torch.nn as nn
import torch.nn.functional as F

from adapter_llm_pipeline import (
    WhisperAdapterLLMPipeline,
    whisper_waveform_to_encoder_windows,
    windows_tensor_to_batch_list,
)
from adapter.cross_attention import QFormerLayer
from adapter.early_commit_gate import EarlyCommitGate
from adapter.rate_controller import AdaptiveRateController
from adapter.stability_buffer import StabilityBuffer
from adapter.streaming_adapter import StreamingAdapter
from adapter.windowing import WhisperFrameWindowizer

print("Imports OK — QFormerLayer module:", QFormerLayer.__module__)


Imports OK — QFormerLayer module: adapter.cross_attention_alt_layers


## KL distillation helper (same idea as stage 3)


In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

def kl_distill(student_logits, teacher_logits, temperature=2.0):
    s = F.log_softmax(student_logits / temperature, dim=-1)
    t = F.softmax(teacher_logits / temperature, dim=-1)
    kl = F.kl_div(s, t, reduction="batchmean")
    return (temperature ** 2) * kl

B, L, V = 1, 20, 1000
student = torch.randn(B, L, V, requires_grad=True)
teacher = torch.randn(B, L, V)
L_kl = kl_distill(student, teacher)
L_kl.backward()
print("L_kl", float(L_kl))


L_kl 20.09909439086914


/tmp/ipykernel_3198916/1630305066.py:16: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  print("L_kl", float(L_kl))


## Gate + adapter reminder


In [4]:
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32
adapter = StreamingAdapter(768, 4096, 4, 2, 4, 2048, 0.0, use_rate_controller=True).to(DEVICE, dtype=DTYPE)
gate = EarlyCommitGate(4096).to(DEVICE, dtype=DTYPE)
print("Built adapter + gate for stage-3-style training loop (wire to two LLMs in your experiment).")


Built adapter + gate for stage-3-style training loop (wire to two LLMs in your experiment).


## Final: mini training loop + checkpoint (sanity)

Stage 3’s full teacher/student loop is expensive; this sanity run checks:
- adapter + gate can be optimized jointly
- KL distillation loss backprop works
- checkpoint save/load works

It uses random student/teacher logits (no LLMs).


In [ ]:
import os
import time
import torch
import torch.nn.functional as F

SAVE_PATH = os.path.join("checkpoints", "adapter_notebook_adapter_gate.pt")
STEPS = 3

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

adapter = StreamingAdapter(
    d_encoder=768, d_llm=4096, num_queries=4, num_layers=2, num_heads=4,
    d_ffn=2048, dropout=0.0, use_rate_controller=True, target_rate=2.0, cross_layer_in_between=1,
).to(DEVICE, dtype=DTYPE)
adapter.train()

gate = EarlyCommitGate(d_llm=4096, hidden_dim=256, threshold=0.5, latency_weight=0.1).to(DEVICE, dtype=DTYPE)
gate.train()

params = list(adapter.parameters()) + list(gate.parameters())
opt = torch.optim.AdamW(params, lr=3e-5)

KL_T = 2.0
LAMBDA_KL = 1.0
LAMBDA_GATE = 0.5


def kl_distill(student_logits, teacher_logits, temperature=2.0):
    s = F.log_softmax(student_logits / temperature, dim=-1)
    t = F.softmax(teacher_logits / temperature, dim=-1)
    return (temperature ** 2) * F.kl_div(s, t, reduction="batchmean")

print("Running", STEPS, "steps... saving to", SAVE_PATH)
losses=[]
t0=time.time()
for step in range(STEPS):
    # Pretend we already have accumulated adapter tokens for a stream
    acc = torch.randn(1, 8, 4096, device=DEVICE, dtype=DTYPE)
    gr = gate(acc, timestep=2, total_timesteps=5)

    # Random logits stand in for student/teacher
    student = torch.randn(1, 32, 1000, device=DEVICE, dtype=torch.float32, requires_grad=True)
    teacher = torch.randn(1, 32, 1000, device=DEVICE, dtype=torch.float32)
    L_kl = kl_distill(student, teacher, temperature=KL_T)

    # Couple adapter params into graph with a tiny regularizer
    reg = sum(p.float().pow(2).mean() for p in adapter.parameters()) * 0.0

    L = LAMBDA_KL * L_kl + LAMBDA_GATE * gr["gate_loss"].float() + reg
    opt.zero_grad()
    L.backward()
    torch.nn.utils.clip_grad_norm_(params, 1.0)
    opt.step()

    losses.append(float(L.detach().cpu()))
    print(f"step {step}: L={losses[-1]:.4f} kl={float(L_kl):.4f} gate={float(gr['gate_loss']):.4f}")

ckpt = {
    "stage": 3,
    "adapter_state_dict": adapter.state_dict(),
    "gate_state_dict": gate.state_dict(),
    "avg_loss": sum(losses)/len(losses),
}
torch.save(ckpt, SAVE_PATH)
print("Saved checkpoint.")
print("Elapsed_s:", time.time() - t0)


## Full training (CLI runner)

Stage 3 full distillation is heavy. This cell runs the official stage script (`training/adapter_task_trainer.py`) using the current notebook kernel’s Python, with `cwd` set to the adapter package root.


In [ ]:
import os
import subprocess
import sys

_script = os.path.join(PROJECT_ROOT, "training", "adapter_task_trainer.py")
subprocess.run([sys.executable, _script], cwd=PROJECT_ROOT, check=False)
